In [ ]:
# Import the required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tifffile

# Define the generator model using Keras
generator = keras.Sequential(
    [
        keras.Input(shape=(100,)),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(1024),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(50 * 50 * 5, activation="tanh"),
        layers.Reshape((50, 50, 5)),
    ],
    name="generator",
)

# Define the discriminator model using Keras
discriminator = keras.Sequential(
    [
        keras.Input(shape=(50, 50, 5)),
        layers.Flatten(),
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="discriminator",
)

# Define the GAN model as a combination of generator and discriminator models
discriminator.trainable = False
gan_input = keras.Input(shape=(100,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name="gan")

# Compile the models
generator_optimizer = keras.optimizers.Adam(learning_rate=0.002, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.002, beta_1=0.5)
discriminator.compile(loss="binary_crossentropy", optimizer=discriminator_optimizer)
gan.compile(loss="binary_crossentropy", optimizer=generator_optimizer)

# Load the .tif images and preprocess them
import os

# Define the root directory where the images are located
root_dir = "./"

# Recursively find all the .tif files in the root directory and its subdirectories
image_paths = []
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("_UNHEALTHY.tif"):
            image_path = os.path.join(dirpath, filename)
            image_paths.append(image_path)
            
images = []
for path in image_paths:
    # Load the TIFF image
    image = tifffile.imread(path)

    # Display the image shape
    print(image.shape)

    #image = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    # Convert the image to a NumPy array and normalize the pixel values to the range [-1, 1]
    #image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 127.5 - 1.0

    # Resize the image to the desired size using OpenCV
    image = cv2.resize(image, (50, 50))

    # Append the preprocessed image to the list of images
    images.append(image)
    
train_images = np.array(images)
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(32)

# Define a function to generate images using the trained GAN model
def generate_images(model, noise, epoch):
    # Generate images from noise using the generator model
    generated_images = model.predict(noise)

    # Rescale pixel values to the range [0, 1]
    generated_images = (1/(2*2.25)) * generated_images + 0.5

    # Save the generated images
    for i in range(generated_images.shape[0]):
        tifffile.imwrite(f"generated_images/{epoch}_{i}.tif", generated_images[i])

# Train the GAN model on the preprocessed dataset
epochs = 1000
noise_dim = 100
num_examples_to_generate = 20
seed = tf.random.normal([num_examples_to_generate, noise_dim])

for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    for real_images in train_dataset:
        # Generate random noise
        noise = tf.random.normal([real_images.shape[0], noise_dim])

        # Generate fake images using the generator model
        fake_images = generator.predict(noise)

        # Concatenate real and fake images
        combined_images = tf.concat([real_images, fake_images], axis=0)

        # Create labels for real and fake images
        real_labels = tf.ones((real_images.shape[0], 1))
        fake_labels = tf.zeros((fake_images.shape[0], 1))
        combined_labels = tf.concat([real_labels, fake_labels], axis=0)

        # Train the discriminator model
        discriminator_loss = discriminator.train_on_batch(combined_images, combined_labels)

        # Train the generator model
        noise = tf.random.normal([real_images.shape[0], noise_dim])
        generator_loss = gan.train_on_batch(noise, real_labels)

    # Generate images using the trained GAN model
    if epoch % 100 == 0:
        generate_images(generator, seed, epoch)

# Save the generator model
generator.save("generator_model.h5")

(312, 461, 5)
(305, 458, 5)
(309, 454, 5)
(307, 457, 5)
(312, 454, 5)
(310, 455, 5)
(316, 458, 5)
(309, 464, 5)
(306, 457, 5)
(322, 465, 5)
(319, 459, 5)
(308, 455, 5)
(318, 453, 5)
(307, 457, 5)
(311, 454, 5)
(307, 461, 5)
(320, 462, 5)
(312, 464, 5)
(315, 462, 5)
(312, 466, 5)
(310, 463, 5)
(304, 456, 5)
(322, 465, 5)
(309, 460, 5)
(310, 456, 5)
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Epoch 30
Epoch 31
Epoch 32
Epoch 33
Epoch 34
Epoch 35
Epoch 36
Epoch 37
Epoch 38
Epoch 39
Epoch 40
Epoch 41
Epoch 42
Epoch 43
Epoch 44
Epoch 45
Epoch 46
Epoch 47
Epoch 48
Epoch 49
Epoch 50
Epoch 51


Epoch 52
Epoch 53
Epoch 54
Epoch 55
Epoch 56
Epoch 57
Epoch 58
Epoch 59
Epoch 60
Epoch 61
Epoch 62
Epoch 63
Epoch 64
Epoch 65
Epoch 66
Epoch 67
Epoch 68
Epoch 69
Epoch 70
Epoch 71
Epoch 72
Epoch 73
Epoch 74
Epoch 75
Epoch 76
Epoch 77
Epoch 78
Epoch 79
Epoch 80
Epoch 81
Epoch 82
Epoch 83
Epoch 84
Epoch 85
Epoch 86
Epoch 87
Epoch 88
Epoch 89
Epoch 90
Epoch 91
Epoch 92
Epoch 93
Epoch 94
Epoch 95
Epoch 96
Epoch 97
Epoch 98
Epoch 99
Epoch 100
Epoch 101
Epoch 102
Epoch 103
Epoch 104


Epoch 105
Epoch 106
Epoch 107
Epoch 108
Epoch 109
Epoch 110
Epoch 111
Epoch 112
Epoch 113
Epoch 114
Epoch 115
Epoch 116
Epoch 117
Epoch 118
Epoch 119
Epoch 120
Epoch 121
Epoch 122
Epoch 123
Epoch 124
Epoch 125
Epoch 126
Epoch 127
Epoch 128
Epoch 129
Epoch 130
Epoch 131
Epoch 132
Epoch 133
Epoch 134
Epoch 135
Epoch 136
Epoch 137
Epoch 138
Epoch 139
Epoch 140
Epoch 141
Epoch 142
Epoch 143
Epoch 144
Epoch 145
Epoch 146
Epoch 147
Epoch 148
Epoch 149
Epoch 150
Epoch 151
Epoch 152
Epoch 153
Epoch 154
Epoch 155
Epoch 156
Epoch 157


Epoch 158
Epoch 159
Epoch 160
Epoch 161
Epoch 162
Epoch 163
Epoch 164
Epoch 165
Epoch 166
Epoch 167
Epoch 168
Epoch 169
Epoch 170
Epoch 171
Epoch 172
Epoch 173
Epoch 174
Epoch 175
Epoch 176
Epoch 177
Epoch 178
Epoch 179
Epoch 180
Epoch 181
Epoch 182
Epoch 183
Epoch 184
Epoch 185
Epoch 186
Epoch 187
Epoch 188
Epoch 189
Epoch 190
Epoch 191
Epoch 192
Epoch 193
Epoch 194
Epoch 195
Epoch 196
Epoch 197
Epoch 198
Epoch 199
Epoch 200
Epoch 201
Epoch 202
Epoch 203
Epoch 204
Epoch 205
Epoch 206
Epoch 207
Epoch 208
Epoch 209
Epoch 210


Epoch 211
Epoch 212
Epoch 213
Epoch 214
Epoch 215
Epoch 216
Epoch 217
Epoch 218
Epoch 219
Epoch 220
Epoch 221
Epoch 222
Epoch 223
Epoch 224
Epoch 225
Epoch 226
Epoch 227
Epoch 228
Epoch 229
Epoch 230
Epoch 231
Epoch 232
Epoch 233
Epoch 234
Epoch 235
Epoch 236
Epoch 237
Epoch 238
Epoch 239
Epoch 240
Epoch 241
Epoch 242
Epoch 243
Epoch 244
Epoch 245
Epoch 246
Epoch 247
Epoch 248
Epoch 249
Epoch 250
Epoch 251
Epoch 252
Epoch 253
Epoch 254
Epoch 255
Epoch 256
Epoch 257
Epoch 258
Epoch 259
Epoch 260
Epoch 261
Epoch 262
Epoch 263


...

Epoch 370
Epoch 371
Epoch 372
Epoch 373
Epoch 374
Epoch 375
Epoch 376
Epoch 377
Epoch 378
Epoch 379
Epoch 380
Epoch 381
Epoch 382
Epoch 383
Epoch 384
Epoch 385
Epoch 386
Epoch 387
Epoch 388
Epoch 389
Epoch 390
Epoch 391
Epoch 392
Epoch 393
Epoch 394
Epoch 395
Epoch 396
Epoch 397
Epoch 398
Epoch 399
Epoch 400
Epoch 401
Epoch 402
Epoch 403
Epoch 404
Epoch 405
Epoch 406
Epoch 407
Epoch 408
Epoch 409
Epoch 410
Epoch 411
Epoch 412
Epoch 413
Epoch 414
Epoch 415
Epoch 416
Epoch 417
Epoch 418
Epoch 419
Epoch 420
Epoch 421
Epoch 422


Epoch 423
Epoch 424
Epoch 425
Epoch 426
Epoch 427
Epoch 428
Epoch 429
Epoch 430
Epoch 431
Epoch 432
Epoch 433
Epoch 434
Epoch 435
Epoch 436
Epoch 437
Epoch 438
Epoch 439
Epoch 440
Epoch 441
Epoch 442
Epoch 443
Epoch 444
Epoch 445
Epoch 446
Epoch 447
Epoch 448
Epoch 449
Epoch 450
Epoch 451
Epoch 452
Epoch 453
Epoch 454
Epoch 455
Epoch 456
Epoch 457
Epoch 458
Epoch 459
Epoch 460
Epoch 461
Epoch 462
Epoch 463
Epoch 464
Epoch 465
Epoch 466
Epoch 467
Epoch 468
Epoch 469
Epoch 470
Epoch 471
Epoch 472
Epoch 473
Epoch 474
Epoch 475


Epoch 476
Epoch 477
Epoch 478
Epoch 479
Epoch 480
Epoch 481
Epoch 482
Epoch 483
Epoch 484
Epoch 485
Epoch 486
Epoch 487
Epoch 488
Epoch 489
Epoch 490
Epoch 491
Epoch 492
Epoch 493
Epoch 494
Epoch 495
Epoch 496
Epoch 497
Epoch 498
Epoch 499
Epoch 500
Epoch 501
Epoch 502
Epoch 503
Epoch 504
Epoch 505
Epoch 506
Epoch 507
Epoch 508
Epoch 509
Epoch 510
Epoch 511
Epoch 512
Epoch 513
Epoch 514
Epoch 515
Epoch 516
Epoch 517
Epoch 518
Epoch 519
Epoch 520
Epoch 521
Epoch 522
Epoch 523
Epoch 524
Epoch 525
Epoch 526
Epoch 527
Epoch 528


Epoch 529
Epoch 530
Epoch 531
Epoch 532
Epoch 533
Epoch 534
Epoch 535
Epoch 536
Epoch 537
Epoch 538
Epoch 539
Epoch 540
Epoch 541
Epoch 542
Epoch 543
Epoch 544
Epoch 545
Epoch 546
Epoch 547
Epoch 548
Epoch 549
Epoch 550
Epoch 551
Epoch 552
Epoch 553
Epoch 554
Epoch 555
Epoch 556
Epoch 557
Epoch 558
Epoch 559
Epoch 560
Epoch 561
Epoch 562
Epoch 563
Epoch 564
Epoch 565
Epoch 566
Epoch 567
Epoch 568
Epoch 569
Epoch 570
Epoch 571
Epoch 572
Epoch 573
Epoch 574
Epoch 575
Epoch 576
Epoch 577
Epoch 578
Epoch 579
Epoch 580
Epoch 581


Epoch 582
Epoch 583
Epoch 584
Epoch 585
Epoch 586
Epoch 587
Epoch 588
Epoch 589
Epoch 590
Epoch 591
Epoch 592
Epoch 593
Epoch 594
Epoch 595
Epoch 596
Epoch 597
Epoch 598
Epoch 599
Epoch 600
Epoch 601
Epoch 602
Epoch 603
Epoch 604
Epoch 605
Epoch 606
Epoch 607
Epoch 608
Epoch 609
Epoch 610
Epoch 611
Epoch 612
Epoch 613
Epoch 614
Epoch 615
Epoch 616
Epoch 617
Epoch 618
Epoch 619
Epoch 620
Epoch 621
Epoch 622
Epoch 623
Epoch 624
Epoch 625
Epoch 626
Epoch 627
Epoch 628
